# MAE 6291 Generative AI for Engineering Research

**Class 6**

## Jupyter AI 

Jupyter AI is an official extension of JupyterLab that interfaces with third-party generative AI models and brings them right into Jupyter notebooks. 

It is an open-source project, and its documentation can be found at https://jupyter-ai.readthedocs.io/

To use Jupyter AI, you need to combine it with an AI model from one of the various providers: OpenAI, Anthropic, Google, etc. Most AI model providers have an associated Python package, and can be accessed via an API key. This allows you to interact with their models programmatically, either from a Python program or in a Jupyter notebook. 

For example, to use OpenAI models programmatically, we would import the `openai` Python package, and set the `OPENAI_API_KEY` environment variable. You first need to generate the API key in your OpenAI account at: http://platform.openai.com

In JupyterLab, with Jupyter AI installed, you set the API keys via the _Settings_ menu, selecting _Jupyternaut settings_. Consult the [Model providers](https://jupyter-ai.readthedocs.io/en/v3/users/index.html#model-providers) section of the Jupyter AI documentation for more details. 

Jupyter AI provides two interaction modes: a **chat interface** for conversational workflows and **magic commands** for programmatic, reproducible AI interactions directly in notebook cells. We'll explore both below.

> Note: As of early 2026, the [GW JupyterHub](http://go.gwu.edu/jupyter) (Engineering Computations image) has installed the `jupyter_ai` library version `3.0.0b9 `.

## Jupyter AI chat interface

The Jupyter AI extension adds a native chat interface right in JupyterLab. Open it by clicking the "chat" icon on the sidebar. It looks like this:

![](chat_icon.png)

A side panel will open, initially blank. At the top, you can start a new chat by clicking on **+Chat**, or you can open a previously saved chat. You give each new chat a _name_, as it will be saved on your home directory with the `.chat` extension. If you will no longer need to keep a record of the particular chat, you can delete that file.

Interacting with AI via the chat interface is a natural way to ask questions, get explanations for code, work through code errors, etc. But Jupyter AI is more than chat: it is an agent that can collaborate with you directly on the open notebook. The agent is called **Jupyternaut** and it can: 

- Explain code and concepts in plain language
- Suggest or generate code snippets and minimal examples you can paste or have added to your notebook
- Edit or add notebook cells for you (when you ask it to) and execute them so you can iterate quickly
- Help debug errors by reading tracebacks and proposing fixes
- Produce plotting and data-exploration examples tailored to your data shape

**When to use chat vs magics:** The chat interface excels when you're exploring ideas, debugging errors interactively, or need the AI to directly manipulate your notebook (insert/edit cells). For workflows you'll repeat or want to document, magic commands (covered later) offer a programmatic alternative.

### Chats are files

One important architectural feature of Jupyter AI is that **chats are files**. You can start multiple independent chats, and each is saved to its own `.chat` file. Later, you can pick it up where you left off at any point by opening the chat, and continuing the conversation, or you can open the chat file into a JupyterLab tab to simply read through it.

### You can add context

To add context to a chat, you can attach a file via the paper-clip button, or you can drag and drop a file. You can also `@`-mention a file to add it in the chat, and you can also drag-and-drop a notebook cell into the chat. These options reduce the amount of annoying copy-and-paste you need to do!

*Example*

Try it yourself. Execute the cells below, then hover with your mouse over the input-line counter for the fourth (last) cell and when you see the cross-hairs, click and drag to the message box of the chat. You'll see that the cell was added as context for the prompt. 

You can `@`-mention Jupyternaut and ask a question in the message box like: "Explain these lines of code, where `beers_styles` is a Pandas dataframe." Hit enter and wait for the reply. 

In [1]:
import pandas as pd

In [2]:
URL = 'http://go.gwu.edu/engcomp2data1'
beers = pd.read_csv(URL)
beers.columns

Index(['Unnamed: 0', 'abv', 'ibu', 'id', 'name', 'style', 'brewery_id',
       'ounces'],
      dtype='object')

Index(['Unnamed: 0', 'abv', 'ibu', 'id', 'name', 'style', 'brewery_id',
       'ounces'],
      dtype='str')

Index(['Unnamed: 0', 'abv', 'ibu', 'id', 'name', 'style', 'brewery_id',
       'ounces'],
      dtype='object')

In [3]:
beers_clean = beers.dropna()
beers_styles = beers_clean.drop(['Unnamed: 0',
                                 'name','brewery_id',
                                 'ounces','id'], axis=1)
beers_styles.head()

,abv,ibu,style
14,0.061,60.0,American Pale Ale (APA)
21,0.099,92.0,American Barleywine
22,0.079,45.0,Winter Warmer
24,0.044,42.0,American Pale Ale (APA)
25,0.049,17.0,Fruit / Vegetable Beer


,abv,ibu,style
14,0.061,60.0,American Pale Ale (APA)
21,0.099,92.0,American Barleywine
22,0.079,45.0,Winter Warmer
24,0.044,42.0,American Pale Ale (APA)
25,0.049,17.0,Fruit / Vegetable Beer


,abv,ibu,style
14,0.061,60.0,American Pale Ale (APA)
21,0.099,92.0,American Barleywine
22,0.079,45.0,Winter Warmer
24,0.044,42.0,American Pale Ale (APA)
25,0.049,17.0,Fruit / Vegetable Beer


In [4]:
style_counts = beers_styles['style'].value_counts()
style_counts = style_counts.sort_index()
style_means = beers_styles.groupby('style').mean()

Then you could ask a follow-up to Jupyternaut—maybe ask it to insert code for making a scatter plot of `abv` versus `ibu` with markers scaled by `style_counts`. You may need to iterate if the suggested code doesn't work immediately, or you want to refine the result. It always helps to be precise with AI models: maybe you will need to add something like: "Since `style_means` is a dataframe, and `style_counts` is a Pandas series, use the Pandas plotting API directly."

Small icons under the code suggested by Jupyternaut will allow you to insert it in your notebook above the active cell, insert it below, replace the selection in a cell, or copy the code to the clipboard to paste it elsewhere. Try all these options!

And if you have a code cell that produces an error, you can drag it to the chat and ask Jupyternaut to fix it for you.

## Jupyter AI Magics

Jupyter AI also gives you the power to prompt your chosen AI model from a notebook cell, using _magics_. Unlike chat, **magics let you document your AI interactions as executable code**—useful for reproducible workflows, comparing model outputs, or integrating AI responses into automated pipelines.

An important advantage: **the `%%ai` magic works anywhere the IPython kernel runs**, including JupyterLab, Jupyter Notebook, Google Colab, and Visual Studio Code. This means your AI-enhanced notebooks are portable across environments.

_Magics_ in Jupyter are special commands. There are two types:

- line magics, using an `%` before them
- cell magics, using `%%`, the command name, and subsequent lines

With Jupyter AI magics you can bring the AI model into a programmatic environment, which means you can fully document your AI-supported workflow.

The cell below will load the `jupyter_ai_magic_commands`, and the next cell configures a default AI model. Subsequent cells that start with the magic `%%ai` and have a text prompt on a new line right below will invoke the model, send your prompt, and capture the response to show it as output of the code cell. Try it!

In [5]:
%load_ext jupyter_ai_magic_commands

/opt/conda/lib/python3.13/site-packages/jupyter_ai_litellm/__init__.py:8: UserWarning: Importing 'jupyter_ai_litellm' outside a proper installation.
  warnings.warn("Importing 'jupyter_ai_litellm' outside a proper installation.")
No `.env` file containing provider API keys found at /home/jovyan/mae6291-2026/.env.                   You can add API keys to the `.env` file via the AI Settings in the JupyterLab UI.


/Applications/anaconda3/envs/jupyter-ai/lib/python3.13/site-packages/jupyter_ai_litellm/__init__.py:8: UserWarning: Importing 'jupyter_ai_litellm' outside a proper installation.
  warnings.warn("Importing 'jupyter_ai_litellm' outside a proper installation.")
No `.env` file containing provider API keys found at /Users/labarba/Desktop/temp/.env.                   You can add API keys to the `.env` file via the AI Settings in the JupyterLab UI.


/opt/conda/lib/python3.13/site-packages/jupyter_ai_litellm/__init__.py:8: UserWarning: Importing 'jupyter_ai_litellm' outside a proper installation.
  warnings.warn("Importing 'jupyter_ai_litellm' outside a proper installation.")
No `.env` file containing provider API keys found at /home/jovyan/mae6291-2026/.env.                   You can add API keys to the `.env` file via the AI Settings in the JupyterLab UI.


In [6]:
%config AiMagics.initial_language_model = "openai/gpt-5-mini"

In [7]:
%%ai
What is the capital of France?

Paris.

The capital of France is Paris.

The capital of France is Paris.

In [8]:
%%ai
What is the linear advection equation?
Show me the equation using LaTeX, delimited by `$`, then explain the physical meaning of each term.

The one-dimensional linear advection equation is
$\displaystyle \frac{\partial u}{\partial t} + c\,\frac{\partial u}{\partial x} = 0.$

Explanation of each term:
- $u(x,t)$: the advected scalar field (e.g., concentration, temperature, tracer) as a function of position $x$ and time $t$.
- $\displaystyle \frac{\partial u}{\partial t}$: the local (Eulerian) time rate of change of $u$ at a fixed position.
- $c$: the constant advection speed (a scalar). Its sign sets the direction of transport (positive $c$ → transport to increasing $x$).
- $\displaystyle \frac{\partial u}{\partial x}$: the spatial gradient of $u$. Multiplying by $c$ gives the rate at which spatial variations are transported past a point.
- The equation as a whole expresses that changes in $u$ at a point are exactly due to pure transport by the velocity $c$ (no sources, sinks, or diffusion).

Remarks:
- In conservation form (flux form) the same equation can be written $\displaystyle \frac{\partial u}{\partial t} + \frac{\partial (c u)}{\partial x}=0$, where $cu$ is the advective flux.
- In multiple dimensions the equation generalizes to $\displaystyle \frac{\partial u}{\partial t} + \mathbf{a}\cdot\nabla u = 0$, with $\mathbf{a}$ the velocity vector and $\nabla u$ the spatial gradient.
- Solution property: $u$ is constant along characteristic lines $x(t)=x_0+ct$, so initial profiles are simply transported at speed $c$.

The one-dimensional linear advection equation is
$ \displaystyle \frac{\partial u}{\partial t} + c\,\frac{\partial u}{\partial x} = 0. $

Explanation of each term:
- $u(x,t)$: the transported scalar quantity (e.g., concentration, temperature) as a function of position $x$ and time $t$.
- $\displaystyle \frac{\partial u}{\partial t}$: the local time rate of change of $u$ at a fixed position.
- $c$: the (constant) advection speed. Its sign and magnitude set the direction and rate at which features in $u$ are carried.
- $\displaystyle \frac{\partial u}{\partial x}$: the spatial gradient of $u$; it measures how $u$ changes with position.
- The equation $=0$ expresses that there are no sources or sinks: changes in $u$ at a point are entirely due to transport by the velocity $c$.

Remarks:
- If $c>0$, disturbances move to increasing $x$; if $c<0$, they move to decreasing $x$.
- Along characteristics $x(t)=x_0+ct$, $u$ is constant, so for initial data $u(x,0)=u_0(x)$ the solution is $u(x,t)=u_0(x-ct)$. (In LaTeX: $u(x,t)=u_0(x-ct)$.)
- In multiple dimensions the equation generalizes to $ \displaystyle \frac{\partial u}{\partial t} + \mathbf{a}\!\cdot\!\nabla u = 0$, where $\mathbf{a}$ is the constant velocity vector.

The one-dimensional linear advection equation (with constant speed) is
$u_t + c\,u_x = 0$,
where $u=u(x,t)$.

A common multi-dimensional form is
$u_t + \mathbf{v}\cdot\nabla u = 0$,

and the usual initial condition is
$u(x,0)=u_0(x)$.

Meaning of each symbol/term:
- $u(x,t)$: the advected scalar field (e.g., concentration, temperature).
- $t$: time.
- $x$ (or $\mathbf{x}$): spatial coordinate(s).
- $u_t=\partial u/\partial t$: local (Eulerian) time rate of change of $u$ at a fixed point in space.
- $u_x=\partial u/\partial x$ (or $\nabla u$): spatial gradient of $u$.
- $c$ (or $\mathbf{v}$): constant advection (transport) speed (or velocity vector).
- $c\,u_x$ (or $\mathbf{v}\cdot\nabla u$): change in $u$ due to transport by the flow.
- The equation $u_t + c\,u_x=0$ says the local time change is exactly balanced by transport; there are no sources, sinks, or diffusion.

Physical interpretation and simple consequence:
- Characteristics satisfy $\mathrm{d}x/\mathrm{d}t = c$, and along each characteristic $u$ is constant: $\mathrm{d}u/\mathrm{d}t=0$.
- For constant $c$ the solution is $u(x,t)=u_0(x-ct)$, so the initial profile moves rigidly at speed $c$ (right if $c>0$, left if $c<0$) without changing shape.

Since we configured the default AI model as `openai/gpt-5-mini` above with the `%config` line magic, you are seeing the response of that particular model each time you send a prompt with the `%%ai` magic. But you can also specify another model as option to the command, as follows:

`%%ai openai/gpt‑5.1`

(Followed by the prompt text below.)
Run `%ai list` to see a list of all available models, or add the provider as an optional argument to see only that subset of models; e.g., `%ai list openai` will show all OpenAI models.

Because you can specify the model as an argument to the `%%ai` command, you are able to experiment with the same prompt in different models, to compare them.

### Include variable values in your prompt

A powerful feature of `%%ai` magic is the ability to template prompts with Python variables using `{}`. This lets you **parameterize prompts**, run the same query with different inputs, or build reusable prompt patterns—critical for systematic experimentation.

You can use curly brackets, `{}`, with a variable name that will be replaced by its value on sending the prompt to the model. This uses _formatted string literals_ in IPython: a way to dynamically create strings that include variables and other Python expressions.

Below, we write a prompt as a Python string variable, then we send it to our default AI model

In [9]:
prompt_advection = "Write Python code to solve the linear advection equation \
using Euler's method for time, and a backward difference scheme for the space \
variable. Use a step function as initial condition."
print(prompt_advection)

Write Python code to solve the linear advection equation using Euler's method for time, and a backward difference scheme for the space variable. Use a step function as initial condition.


Write Python code to solve the linear advection equation using Euler's method for time, and a backward difference scheme for the space variable. Use a step function as initial condition.


Write Python code to solve the linear advection equation using Euler's method for time, and a backward difference scheme for the space variable. Use a step function as initial condition.


In [10]:
%%ai
{prompt_advection}

Below is a simple, self-contained Python script that solves the 1D linear advection equation using explicit (forward) Euler in time and a backward difference for the spatial derivative. The domain is periodic and the initial condition is a step function.

Key scheme (implemented):
u_i^{n+1} = u_i^n - c * (dt/dx) * (u_i^n - u_{i-1}^n)

Note: For stability you must satisfy the CFL condition |c| * dt / dx <= 1 (choose CFL < 1).

Code:

```python
import numpy as np
import matplotlib.pyplot as plt

def step_initial_condition(x, x0=0.25, x1=0.75):
    """Step function: 1 on [x0,x1), 0 elsewhere."""
    return np.where((x >= x0) & (x < x1), 1.0, 0.0)

def advect_euler_backward(c=1.0, nx=200, CFL=0.8, t_final=1.0):
    """
    Solve u_t + c u_x = 0 using forward Euler in time and backward difference in space.
    Periodic boundary conditions are used.
    """
    # Grid
    L = 1.0
    dx = L / nx
    x = np.linspace(0.0, L, nx, endpoint=False)

    # Time step from CFL
    dt = CFL * dx / abs(c) if c != 0 else 1e-6
    nt = int(np.ceil(t_final / dt))
    dt = t_final / nt  # adjust dt so final time is exactly t_final
    sigma = c * dt / dx  # nondimensional number

    # Initial condition
    u = step_initial_condition(x, x0=0.25, x1=0.75)

    # Store snapshots for plotting
    snapshots = [u.copy()]
    times = [0.0]
    snapshot_interval = max(1, nt // 8)  # store ~8 snapshots

    # Time-stepping loop
    for n in range(1, nt + 1):
        # backward difference: (u_i - u_{i-1}) / dx
        u_prev = np.roll(u, 1)  # u_{i-1} with periodic BC
        u = u - sigma * (u - u_prev)  # explicit update

        if n % snapshot_interval == 0 or n == nt:
            snapshots.append(u.copy())
            times.append(n * dt)

    return x, snapshots, times, sigma

if __name__ == "__main__":
    c = 1.0
    nx = 200
    CFL = 0.8
    t_final = 1.0

    x, snapshots, times, sigma = advect_euler_backward(c=c, nx=nx, CFL=CFL, t_final=t_final)

    # Plot snapshots
    plt.figure(figsize=(8, 5))
    for u, t in zip(snapshots, times):
        plt.plot(x, u, label=f"t={t:.3f}")
    plt.xlabel("x")
    plt.ylabel("u")
    plt.title(f"Linear advection (Euler time + backward space), c={c}, sigma={sigma:.3f}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
```

Notes and remarks:
- The scheme implemented is stable for |sigma| = |c| dt / dx <= 1 (CFL ≤ 1); choose CFL < 1 (e.g., 0.8).
- Using a backward difference with forward Euler is the upwind method when c > 0. If c < 0 you should use a forward difference (downwind for c>0 would be unstable).
- Because this is pure advection with a discontinuous initial condition, numerical dispersion and smearing (numerical diffusion) are expected. The first-order upwind scheme is diffusive. To reduce diffusion you can increase resolution or use higher-order schemes.

Below is a simple, self-contained Python script that solves the 1D linear advection equation
∂u/∂t + c ∂u/∂x = 0
using forward (explicit) Euler in time and a backward difference in space (i.e., u_x ≈ (u_i - u_{i-1})/dx). It uses a step function as the initial condition and periodic boundary conditions. Note: for stability with this scheme you should satisfy the CFL condition nu = c * dt / dx ≤ 1 (typically choose nu ≤ 0.9).

Save as a .py file and run, or run inside a notebook cell.

```python
import numpy as np
import matplotlib.pyplot as plt

# Parameters
L = 1.0            # domain length
nx = 200           # number of grid points
dx = L / nx
x = np.linspace(0, L, nx, endpoint=False)

c = 1.0            # advection speed (assumed > 0 for backward difference to be upwind)
CFL = 0.8          # Courant number (nu = c*dt/dx); must be <= 1 for stability
dt = CFL * dx / c
T = 0.5            # final time
nt = int(np.ceil(T / dt))
dt = T / nt        # adjust dt so nt*dt = T exactly
nu = c * dt / dx

print(f"nx={nx}, dx={dx:.5g}, dt={dt:.5g}, nt={nt}, nu={nu:.5g}")

# Initial condition: step function
u0 = np.zeros_like(x)
# define step between x=a and x=b
a, b = 0.1 * L, 0.3 * L
u0[(x >= a) & (x < b)] = 1.0

u = u0.copy()

# Time integration (forward Euler in time, backward difference in space)
for n in range(nt):
    # backward difference: (u_i - u_{i-1})/dx
    u_im1 = np.roll(u, 1)              # periodic BC: u_{-1} -> last cell
    u = u - nu * (u - u_im1)           # vectorized update

# exact solution (for pure advection with periodic BC): shift of initial profile
shift = (c * T) % L
x_shifted = (x - shift) % L
u_exact = np.zeros_like(x)
u_exact[(x_shifted >= a) & (x_shifted < b)] = 1.0

# Plot
plt.figure(figsize=(8,4))
plt.plot(x, u0, '--', label='Initial')
plt.plot(x, u,  '-', lw=2, label='Numerical (Euler + backward diff)')
plt.plot(x, u_exact, ':', label='Exact (shifted)')
plt.xlabel('x')
plt.ylabel('u')
plt.legend()
plt.title(f'Linear advection at t={T:.3f}, nu={nu:.3f}')
plt.grid(True)
plt.show()
```

Notes and remarks:
- The backward difference used here is upwind for c > 0 (good for stability). If c < 0 you'd need to use a forward difference (use np.roll(u, -1) accordingly).
- This scheme is first-order accurate in both time and space and will introduce numerical diffusion (the step will smear over time).
- Periodic boundary conditions are implemented via np.roll. If you want non-periodic BCs, modify the treatment at boundaries accordingly.

Below is a simple 1D solver for the linear advection equation u_t + c u_x = 0 using forward (explicit Euler) in time and a backward difference in space. The code uses a step function as the initial condition and periodic boundary conditions implemented with numpy.roll.

Important notes:
- The scheme implemented is u_i^{n+1} = u_i^n - nu * (u_i^n - u_{i-1}^n) with nu = c*dt/dx (backward difference). For stability you need the CFL condition |nu| <= 1. In practice this scheme is stable when c > 0 and 0 <= nu <= 1 (if c < 0 you should use a forward difference or an upwind scheme).
- The code below assumes c > 0. If you want to advect left (c < 0) either change the spatial differencing to forward difference or use an upwind switch (I remark a small alternate version below).

Code:

```python
import numpy as np
import matplotlib.pyplot as plt

# Parameters
L = 1.0          # domain length
nx = 200         # number of grid points
x = np.linspace(0, L, nx, endpoint=False)
dx = x[1] - x[0]

c = 1.0          # advection speed (set c>0 for backward-difference stability)
CFL = 0.8        # desired CFL number (must satisfy |c|*dt/dx <= 1)
T = 1.0          # final time

dt = CFL * dx / abs(c)
nt = int(np.ceil(T / dt))
dt = T / nt      # adjust dt so nt*dt = T (keeps CFL approx <= chosen CFL)
nu = c * dt / dx

print("dx =", dx, "dt =", dt, "nu =", nu, "nt =", nt)

# Initial condition: step function (1 on [x0,x1), else 0)
x0, x1 = 0.1 * L, 0.3 * L
u0 = np.where((x >= x0) & (x < x1), 1.0, 0.0)

# Time integration (forward Euler in time, backward difference in space)
u = u0.copy()
for n in range(nt):
    # backward difference: (u_i - u_{i-1})/dx implemented with np.roll for periodic BCs
    u = u - nu * (u - np.roll(u, 1))

# Plot initial and final solution
plt.figure(figsize=(8,4))
plt.plot(x, u0, label='initial', lw=2)
plt.plot(x, u, label=f't = {T:.3f}', lw=2)
plt.xlabel('x')
plt.ylabel('u')
plt.legend()
plt.title('Linear advection: forward Euler in time + backward difference in space')
plt.grid(True)
plt.show()
```

If you want the code to automatically use the appropriate upwind direction (stable for either sign of c), replace the time-stepping loop with:

```python
for n in range(nt):
    if c >= 0:
        # backward difference (use upstream cell i-1)
        u = u - nu * (u - np.roll(u, 1))
    else:
        # forward difference (use upstream cell i+1 when c < 0)
        u = u - nu * (np.roll(u, -1) - u)
```

This upwind switch gives a stable first-order upwind scheme for both c>0 and c<0 (with |nu| <= 1).

In this case, the suggested code is part of the cell output, and if we want to run it, we would need to copy it and paste it into a code cell. If instead you had asked the question in the chat interface, you could have inserted it in a code cell with one click. It does seem that the chat interface offers a better user experience. But you have an option to format the output as code, using: `%%ai --format code` and this should insert the code in a new cell, ready to run.

Note that the default is for Jupyter AI to send the 4 previous human/AI interactions as context to each prompt. You can reset the history with the `%ai reset` command, or you can change the default number of interactions to include with `%config AiMagics.max_history = 2`, for example.

In [11]:
%ai reset

In [12]:
%%ai --format code
{prompt_advection}

In [ ]:
Below is a simple, self-contained Python script that solves the 1D linear advection equation
u_t + c u_x = 0
with explicit (forward) Euler in time and a backward difference (first-order) in space:
u_i^{n+1} = u_i^n - c*(dt/dx)*(u_i^n - u_{i-1}^n).

The code uses a step function as the initial condition and periodic boundary conditions so you can see the step advect around the domain. Note: this backward-difference scheme is the standard upwind scheme when c > 0 and is stable if the CFL number |c|*dt/dx <= 1. If c < 0 you would need a forward difference (or use upwinding properly).

Save to a file (e.g. advect_euler_backward.py) and run.

```python
import numpy as np
import matplotlib.pyplot as plt

def advect_euler_backward(c, L=1.0, nx=200, dt=0.002, t_final=0.5, x0=0.25, x1=0.5, plot_interval=0.05):
    """
    Solve u_t + c u_x = 0 with forward Euler in time and backward difference in space.
    Periodic boundary conditions. Initial condition: step function (1 for x in [x0,x1), 0 elsewhere).
    Parameters:
        c : advection speed
        L : domain length (0..L)
        nx: number of spatial points
        dt : time step
        t_final : final time
        x0, x1 : step initial condition interval
        plot_interval : time between plotted snapshots
    """
    dx = L / nx
    x = np.linspace(0, L, nx, endpoint=False)  # cell-centers
    # initial condition: step function
    u = np.where((x >= x0) & (x < x1), 1.0, 0.0)

    # CFL check
    cfl = abs(c) * dt / dx
    if cfl > 1.0:
        print("WARNING: CFL = {:.3f} > 1. Scheme may be unstable.".format(cfl))
    else:
        print("CFL = {:.3f}".format(cfl))

    nt = int(np.ceil(t_final / dt))
    times = np.linspace(0, t_final, nt+1)

    # Plot initial state
    plt.figure(figsize=(8, 4))
    plt.plot(x, u, label=f"t = 0.0")
    next_plot_time = plot_interval
    current_time = 0.0

    for n in range(nt):
        # backward difference in space with periodic BC implemented by np.roll
        du_dx = (u - np.roll(u, 1)) / dx  # backward difference (u_i - u_{i-1})/dx

        # forward Euler time step
        u = u - c * dt * du_dx

        current_time += dt

        # plot snapshots at intervals
        if current_time >= next_plot_time - 1e-12:
            plt.plot(x, u, label=f"t = {current_time:.3f}")
            next_plot_time += plot_interval

    plt.xlabel("x")
    plt.ylabel("u")
    plt.title(f"Advection (Euler time + backward diff) c={c}, dx={dx:.4f}, dt={dt:.4f}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return x, u

if __name__ == "__main__":
    # Example usage
    c = 1.0         # advection speed
    L = 1.0         # domain length
    nx = 200        # grid points
    dx = L / nx
    dt = 0.4 * dx / abs(c)  # choose dt from CFL (<=1). Here CFL=0.4
    t_final = 1.0

    advect_euler_backward(c, L=L, nx=nx, dt=dt, t_final=t_final, x0=0.1, x1=0.3, plot_interval=0.25)
```

Notes:
- The core update is u = u - c*dt*(u - roll(u,1))/dx.
- For c > 0 the backward difference is the upwind scheme and first-order accurate in space and time.
- The method is stable only when the CFL condition |c|*dt/dx <= 1 is satisfied. If c < 0, using backward differences is not upwind and will be unstable; use a forward difference (or choose upwind dynamically) in that case. If you want a version that automatically uses the upwind direction for any sign of c, I can provide that too.

### Loop Over AI Prompts?

Running AI magics in a loop lets you:
- Ask the same question to multiple models and compare answers
- Apply the same analysis to a list of topics (e.g., explain 10 equations)
- Build datasets of AI responses for further processing

The technique requires capturing outputs programmatically, which we'll demonstrate below.

### Running Cell Magics in a Loop

To call a cell magic repeatedly, use `get_ipython().run_cell_magic(magic_name, options, code)`:
- **`magic_name`**: The magic without `%%` (e.g., `'ai'`, `'timeit'`)
- **`options`**: Arguments after the magic name (or `''` for none)
- **`code`**: The cell body as a string

Example with `%%timeit`:

In [13]:
# This is a loop that runs the %%timeit magic on some code
for i in range(3):
    get_ipython().run_cell_magic('timeit', '', 'x = i + i')

42 ns ± 0.815 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


42.7 ns ± 0.555 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


42.6 ns ± 0.696 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


42.4 ns ± 1.43 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


42.6 ns ± 0.757 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


42.2 ns ± 0.605 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


If you'd like to save the output of each call to `get_ipython().run_cell_magic()` into a file, one possible way is to use the [`%%capture`](http://ipython.readthedocs.io/en/stable/interactive/magics.html#cellmagic-capture) cell magic. We need to give it the name of the variable in which to store output. Since we will call it in a loop, this magic also has to be enclosed in the `get_ipython().run_cell_magic()` command. We can save the third argument in a string variable to make the code more readable.

We could then use the [`%store`](https://ipython.readthedocs.io/en/stable/config/extensions/storemagic.html) line magic to append the output to a file with the shell redirection `>>`, as follows:

In [14]:
run_it = "get_ipython().run_cell_magic('timeit', '', 'x = i + i')"

# This is a modified loop that saves the output of each call into a file
for i in range(3):
    # Capture the standard output and store it in timeit_output
    get_ipython().run_cell_magic('capture','timeit_output', run_it)
    %store timeit_output.stdout >>timeit.txt # Append the output to timeit.txt

Writing 'timeit_output.stdout' (str) to file 'timeit.txt'.


Writing 'timeit_output.stdout' (str) to file 'timeit.txt'.


Writing 'timeit_output.stdout' (str) to file 'timeit.txt'.


Writing 'timeit_output.stdout' (str) to file 'timeit.txt'.


Writing 'timeit_output.stdout' (str) to file 'timeit.txt'.


Writing 'timeit_output.stdout' (str) to file 'timeit.txt'.


### Saving AI Outputs in a Loop

To collect AI responses, we need to **capture** the output of each `%%ai` call. Jupyter's `%%capture` magic stores output in a variable. After experimenting, I found AI responses are stored in the `.outputs[0].data['text/markdown']` attribute.

Here's the pattern: we'll ask the AI to explain three different equations and collect all responses in a list.

In [15]:
equations = ['advection', 'diffusion', 'Burgers']

In [16]:
preprompt = "Explain the equation listed below. \
Show me the equation using LaTeX, delimited by `$`, \
then explain the physical meaning of each term."

In [17]:
explanations = []

for equation in equations:
    prompt = f"{preprompt}\n ''' {equation} '''"
    run_it = f"get_ipython().run_cell_magic('ai', '', prompt)"
    get_ipython().run_cell_magic('capture','ai_output', run_it)
    explanations.append(ai_output.outputs[0].data['text/markdown'])

In [18]:
print(explanations[0])

The 1D linear advection equation is
$u_t + c\,u_x = 0$.

Term-by-term:
- $u(x,t)$: the transported scalar (e.g., concentration, temperature, tracer) as a function of space and time.
- $u_t \equiv \frac{\partial u}{\partial t}$: local (Eulerian) rate of change of $u$ at a fixed position — how the value at a point changes in time.
- $u_x \equiv \frac{\partial u}{\partial x}$: spatial gradient of $u$ — how $u$ varies in space.
- $c$: constant advection speed (positive means transport to increasing $x$).
- $c\,u_x$: the advective term — the change in $u$ due to transport by a flow with speed $c$.
- The equation $u_t + c\,u_x = 0$ states that the local time change is exactly balanced by transport, so the quantity is neither created nor destroyed but simply moved. Equivalently, $u$ is constant along characteristic lines given by $\frac{dx}{dt}=c$, so $u(x,t)=u_0(x-ct)$ (initial profile shifted at speed $c$).

In multiple dimensions the same idea is written
$u_t + \mathbf{v}\cdot\nabla u = 0$

$u_t + c\,u_x = 0$

(1D linear advection / transport equation)

Explanation of each symbol and term:

- $u(x,t)$: the transported quantity (a scalar field) — e.g. concentration of a pollutant, temperature, or a passive tracer — evaluated at position $x$ and time $t$.

- $u_t \equiv \frac{\partial u}{\partial t}$: the local (Eulerian) time derivative of $u$ at fixed $x$. It measures how the value of $u$ at a given point changes in time.

- $c$: the advection speed (a given constant in this form). Its sign and magnitude set the direction and speed at which features of $u$ move: $c>0$ moves features to the right, $c<0$ to the left.

- $u_x \equiv \frac{\partial u}{\partial x}$: the spatial gradient of $u$. Multiplying by $c$ gives the rate at which spatial variation of $u$ is carried past a fixed point.

- $c\,u_x$: the advective flux derivative term; together with $u_t$ it expresses that temporal change at a point is caused by the advection of spatial gradients.

- $0$ on the right-hand 

In [19]:
from IPython.display import display, Markdown

In [20]:
Markdown(explanations[0])

The 1D linear advection equation is
$u_t + c\,u_x = 0$.

Term-by-term:
- $u(x,t)$: the transported scalar (e.g., concentration, temperature, tracer) as a function of space and time.
- $u_t \equiv \frac{\partial u}{\partial t}$: local (Eulerian) rate of change of $u$ at a fixed position — how the value at a point changes in time.
- $u_x \equiv \frac{\partial u}{\partial x}$: spatial gradient of $u$ — how $u$ varies in space.
- $c$: constant advection speed (positive means transport to increasing $x$).
- $c\,u_x$: the advective term — the change in $u$ due to transport by a flow with speed $c$.
- The equation $u_t + c\,u_x = 0$ states that the local time change is exactly balanced by transport, so the quantity is neither created nor destroyed but simply moved. Equivalently, $u$ is constant along characteristic lines given by $\frac{dx}{dt}=c$, so $u(x,t)=u_0(x-ct)$ (initial profile shifted at speed $c$).

In multiple dimensions the same idea is written
$u_t + \mathbf{v}\cdot\nabla u = 0$,

where $\mathbf{v}(\mathbf{x},t)$ is the velocity field and $\mathbf{v}\cdot\nabla u$ is the directional derivative of $u$ along the flow. The conservative (divergence) form is
$u_t + \nabla\cdot(\mathbf{v}u) = 0$,

which is equivalent to the previous form when $\nabla\cdot\mathbf{v}=0$ (incompressible flow). Physically, the advection equation models pure transport (no diffusion or sources/sinks).

$u_t + c\,u_x = 0$

(1D linear advection / transport equation)

Explanation of each symbol and term:

- $u(x,t)$: the transported quantity (a scalar field) — e.g. concentration of a pollutant, temperature, or a passive tracer — evaluated at position $x$ and time $t$.

- $u_t \equiv \frac{\partial u}{\partial t}$: the local (Eulerian) time derivative of $u$ at fixed $x$. It measures how the value of $u$ at a given point changes in time.

- $c$: the advection speed (a given constant in this form). Its sign and magnitude set the direction and speed at which features of $u$ move: $c>0$ moves features to the right, $c<0$ to the left.

- $u_x \equiv \frac{\partial u}{\partial x}$: the spatial gradient of $u$. Multiplying by $c$ gives the rate at which spatial variation of $u$ is carried past a fixed point.

- $c\,u_x$: the advective flux derivative term; together with $u_t$ it expresses that temporal change at a point is caused by the advection of spatial gradients.

- $0$ on the right-hand side: indicates no sources, sinks, or diffusion — the quantity is conserved and only transported by the flow.

Physical meaning / interpretation: the equation states that the value of $u$ is constant along characteristic curves defined by $\frac{dx}{dt}=c$. Equivalently, if you follow a fluid parcel moving with speed $c$, $u$ carried by that parcel does not change in time:
$\displaystyle \frac{d}{dt}u(x(t),t)=0$ for $dx/dt=c$.
Thus an initial profile is simply translated at speed $c$ without changing shape (in the idealized, non-diffusive case).

Vector/general form (for a spatially varying velocity field $\mathbf v(x,t)$):
$u_t + \mathbf v\!\cdot\!\nabla u = 0.$
If written in conservative form, $u_t + \nabla\!\cdot(\mathbf v\,u)=0$, care is needed because the two forms differ when $\nabla\!\cdot\mathbf v\neq0$ (the conservative form enforces conservation of the integral of $u$). Examples of use: pollutant transport by a given flow, passive scalar advection, wave propagation in a mean flow.

## Summary

You've learned three ways to integrate AI into Jupyter workflows:

1. **Chat Interface**: Interactive, collaborative agent that can edit notebooks
   - Best for: exploration, debugging, getting code suggestions
   
2. **`%%ai` Magic**: Inline AI prompts as executable cells
   - Best for: documented workflows, comparing models, single queries
   
3. **Programmatic Looping**: Run AI magics with `get_ipython().run_cell_magic()`
   - Best for: batch processing, systematic comparisons, building datasets

**Next Steps**: Try using Jupyternaut to help with your own research code—ask it to explain a function, debug an error, or generate a visualization!

## Supplementary Materials

Consider following these online courses from the DeepLearning.AI platform:

- [AI Python for Beginners](https://www.deeplearning.ai/short-courses/ai-python-for-beginners/)
- [Jupyter AI: AI Coding in Notebooks](https://www.deeplearning.ai/short-courses/jupyter-ai-coding-in-notebooks/)
